# Extending our use of agents to include tools!

Tools are external resources that enable agents to perform actions beyond simple text generation.  For example:
* Search Tools
  * Fetch real-time information from sources like Google, Bing, DuckDuckGo, Wikipedia, or ArXiv. Essential for up-to-date facts not in the LLM’s training data. 
* Calculation Tools
  * Handle precise mathematical operations. Includes LLMMathChain (uses an LLM to determine the calculation, then a reliable calculator) and the Python REPL Tool (executes Python code -> *use with caution due to security risks*). 
* Database Tools
  * Query SQL databases using tools like SQLDatabaseToolkit or QuerySQLDataBaseTool, enabling natural language questions about structured data. 
* API Interaction Tools
  * Interface with external APIs (e.g., weather, stock prices) via generic HTTP wrappers or custom tools. 
* File System Tools
  * Read, write, or list files on the local filesystem -> *also requires careful security consideration*.
* Web Browsing Tools
  * Load and extract content from web pages using libraries like requests or BeautifulSoup. 
* Custom Tools
  * Developers can wrap any Python function into a tool using a docstring, enabling integration with proprietary logic, internal services, or domain-specific workflows.

In [ ]:
from langchain.tools import tool

Let's create our own:

In [ ]:
def tool1(x):
    return x ** 0.5

In [ ]:
tool1(300)

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

Initialize our model:

In [ ]:
from langchain_openai import ChatOpenAI

# This assumes that you have a "keys.py" file in this directory
# with NRP_TOK assigned the value of your NRP API token (required)
import keys
NRP_TOK = keys.NRP_TOK

nrp_llm_url = "https://ellm.nrp-nautilus.io/v1"

model = ChatOpenAI(model = 'gpt-oss',
                   api_key = NRP_TOK,
                   base_url = nrp_llm_url,
                   use_responses_api=False)

Initialize our agent:

In [ ]:
# We'll have to iterate on this
agent = create_agent(model=model,
                     tools=[tool1])

In [ ]:
# We'll have to iterate on this

def tool1(x):
    "Calculates the square root of a number"
    return x ** 0.5
    
agent = create_agent(model=model,
                     tools=[tool1])

In [ ]:
question = HumanMessage(content="What is the square root of 467?")

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

In [ ]:
tool1(467)

... we can't actually be sure from the output what the agent did.  However:

In [ ]:
response

We can be a little better about explicitly coding our function as a model tool:

In [ ]:
@tool
def tool1(x):
    "Calculates the square root of a number"
    return x ** 0.5

In [ ]:
tool1.invoke({"x": 467})

The tool has:
* name
* description
* parameters

These can be automatically grabbed from function names and docstrings.

Or they can be more explicitly specified:

In [ ]:
tool1.name

In [ ]:
tool1.description

In [ ]:
tool1.args

In [ ]:
@tool("square_root", 
      description="Calculate the square root of a number")
def tool1(a):
    return a ** 0.5

In [ ]:
tool1.name, tool1.description, tool1.args

Try running the below a couple times -- note that the agent may not always decide to call the tool!

In [ ]:
agent = create_agent(model=model,
                     tools=[tool1])

In [ ]:
question = HumanMessage(content="What is the square root of 330?")

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)
print('TOOL CALL:', response['messages'][1].tool_calls)
response

# The book example

In [ ]:
from langchain_community.tools import DuckDuckGoSearchResults

In [ ]:
from langchain_community.agent_toolkits.load_tools import load_tools, Tool
from langchain_community.tools import DuckDuckGoSearchResults

search = DuckDuckGoSearchResults()

search_tool = Tool(
    name="duckduck",
    description="A web search engine. Use this to as a search engine for general queries.",
    func=search.run,
)

#tools = load_tools(["llm-math"], llm=openai_llm)
# ... Using our own LLM model to do math...
tools = load_tools(["llm-math"], llm=model)

tools.append(search_tool)

In [ ]:
from langchain_core.prompts import PromptTemplate

In [ ]:
react_template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

prompt = PromptTemplate(
    template=react_template,
    input_variables=["tools", "tool_names", "input", "agent_scratchpad"]
)

In [ ]:
from langchain_classic.agents import AgentExecutor, create_react_agent

# agent = create_react_agent(openai_llm, tools, prompt)
agent = create_react_agent(model, tools, prompt)

agent_executor = AgentExecutor(
    agent=agent, tools=tools, verbose=True, handle_parsing_errors=True
)

In [ ]:
# What is the Price of a MacBook Pro?
agent_executor.invoke(
    {
        "input": "What is the current price of a MacBook Pro in USD? How much would it cost in EUR if the exchange rate is 0.85 EUR for 1 USD?"
    }
)

Let's recast this a bit:

In [ ]:
duckduck_client = DuckDuckGoSearchResults()

@tool("duckduck",
      description="A web search engine. Use this tool as a search engine for general queries.")
def web_search(query):
    results = duckduck_client.run(query)
    return results

In [ ]:
web_search.invoke("Who is the current mayor of San Francisco?")

In [ ]:
@tool
def llm_math(query):
    "Do a mathematical calculation."
    return model.invoke(f"Return the mathematical result of executing {query}.  Return only a number.")

In [ ]:
llm_math.invoke("1,200 * 4.5")

What is the previous prompt template?  -> a system message

In v1:

* The "scratchpad" lives in the agent state as a list of messages (HumanMessage, AIMessage, ToolMessage, ...).
* `create_agent` automatically sends those messages to the model each turn.
* You can reconstruct or summarize that scratchpad in middleware or on the output

In [ ]:
agent = create_agent(model=model,
                     tools=[llm_math, web_search])

In [ ]:
from IPython.display import display, Markdown

In [ ]:
question = HumanMessage(content='''
What is the current price of a MacBook Pro in USD? 
How much would it cost in EUR if the exchange rate is 0.85 EUR for 1 USD?
''')

response = agent.invoke(
    {"messages": [question]}
)

text = response['messages'][-1].content
display(Markdown(text))

In [ ]:
response

In [ ]:
for i in response['messages']:
    try:
        print(i.tool_calls)
    except:
        print('No tool call')

In [ ]:
from langchain.messages import AIMessage, HumanMessage, ToolMessage

def pretty_react_trace(messages):
    lines = []
    for msg in messages:
        if isinstance(msg, HumanMessage):
            lines.append(f"Question: {msg.content}")
        elif isinstance(msg, AIMessage):
            if msg.content:
                lines.append(f"Thought: {msg.content}")
            # Any tool calls here are the "Action" + "Action Input"
            for tc in msg.tool_calls:
                lines.append(f"Action: {tc['name']}")
                lines.append(f"Action Input: {tc['args']}")
        elif isinstance(msg, ToolMessage):
            lines.append(f"Observation: {msg.content}")
    return "\n".join(lines)

In [ ]:
messages = response['messages']
print(pretty_react_trace(messages))
print('-'*80)
print("\nFinal Answer:", messages[-1].content)